# PEP 8 Demo — **AFTER** (Corrected Code)

This notebook fetches Pokémon data from the free
[PokéAPI](https://pokeapi.co/) and analyzes base stats.

Same logic as **`pep8_before.ipynb`**, rewritten to follow
[PEP 8](https://peps.python.org/pep-0008/).
Each section explains what was fixed.

---
## Setup

**Fixes:**
- One `import` per line
- Standard-library imports first, then third-party, separated by a blank line
- Alphabetical order within each group
- No inline comments on import lines — they add clutter, not clarity

In [ ]:
import json
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

---
## Configuration

**Fixes:**
- Module-level constants in `UPPER_SNAKE_CASE`
- Spaces around `=`
- Inline comments: two spaces before `#`, one space after
- Long lists broken across lines — one item per line

In [ ]:
BASE_URL = "https://pokeapi.co/api/v2/"  # API root
NUM_POKEMON = 50  # how many to fetch
STAT_NAMES = [
    "hp",
    "attack",
    "defense",
    "special-attack",
    "special-defense",
    "speed",
]

---
## Fetching data from the API

We call the `/pokemon/{id}` endpoint for each Pokémon.
The API returns JSON with stats, types, height, weight, etc.

**Fixes:**
- Function name in `snake_case`
- Multi-line docstring replaces the `##` block comment
- Spaces around `=`, `!=`, `+`
- Inline comments kept **short and useful** (explain *why*, not *what*)
- Long dictionary literal broken across lines with trailing comma
- Blank lines separate logical steps inside the function

In [ ]:
def fetch_pokemon(pokemon_id):
    """
    Fetch one Pokémon by ID.

    Returns a dict with name, types, and base stats,
    or None if the request fails.
    """
    url = BASE_URL + f"pokemon/{pokemon_id}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Error fetching id {pokemon_id}: "
              f"{response.status_code}")
        return None

    data = response.json()

    # Extract the fields we care about
    stats = {
        s["stat"]["name"]: s["base_stat"]
        for s in data["stats"]
    }
    types = [t["type"]["name"] for t in data["types"]]

    return {
        "name": data["name"],
        "id": data["id"],
        "types": types,
        "height": data["height"],
        "weight": data["weight"],
        **stats,  # merge base stats into the dict
    }

**Fixes in the fetch loop below:**
- `is not None` instead of `!= None`
- `snake_case` variable names
- Spaces around `%`, `==`, `+`
- Comment on its own line, not crammed after code

In [ ]:
all_pokemon = []

for i in range(1, NUM_POKEMON + 1):
    result = fetch_pokemon(i)
    if result is not None:
        all_pokemon.append(result)

    # Progress update every 10 fetches
    if i % 10 == 0:
        print(f"Fetched {i}/{NUM_POKEMON}...")

print(f"Done! Got {len(all_pokemon)} pokemon.")

---
## Build a DataFrame

Convert the list of dicts to a pandas DataFrame for easier analysis.

**Fixes:**
- Inline comments explain *why* (unit conversion rationale), not *what*
- Consistent double-quote strings

In [ ]:
df = pd.DataFrame(all_pokemon)
df["total"] = df[STAT_NAMES].sum(axis=1)
df["primary_type"] = df["types"].apply(lambda t: t[0])
df["height_m"] = df["height"] / 10  # API gives decimeters
df["weight_kg"] = df["weight"] / 10  # API gives hectograms

print(f"DataFrame shape: {df.shape}")
df.head(10)

---
## Analysis class

A reusable class that computes summary statistics
for a given type of Pokémon.

**Fixes:**
- Class name in `PascalCase`
- Instance attributes in `snake_case`
- `is None` instead of `== None`
- One blank line between methods
- Multi-line docstrings replace `##` block comments
- No semicolons — one statement per line
- Space after `:` in return dictionaries

In [ ]:
class PokemonAnalyzer:
    """
    Analyze a subset of Pokémon filtered by type.

    Provides methods for stat summaries and rankings.
    """

    def __init__(self, dataframe, type_filter=None):
        self.type_name = type_filter

        if type_filter is None:
            self.data = dataframe.copy()
        else:
            mask = dataframe["primary_type"] == type_filter
            self.data = dataframe[mask].copy()

        self.count = len(self.data)

    def stat_summary(self):
        """Return mean of each base stat as a dict."""
        if self.count == 0:
            return None
        return (
            self.data[STAT_NAMES]
            .mean()
            .round(1)
            .to_dict()
        )

    def top_n(self, stat="total", n=5):
        """Return the top N Pokémon by a given stat."""
        return (
            self.data
            .nlargest(n, stat)[["name", stat]]
            .reset_index(drop=True)
        )

    def print_report(self):
        """Print a formatted summary to stdout."""
        label = self.type_name if self.type_name else "All Types"
        print(f"--- {label} ({self.count} pokemon) ---")

        stats = self.stat_summary()
        if stats is None:
            print("  No data")
            return

        for stat_name, val in stats.items():
            print(f"  {stat_name}: {val}")

---
## Run the analysis

Print stat summaries for all Pokémon and for each primary type.

**Fixes:**
- Descriptive variable names (`unique_types`, not `uniqueTypes`)
- Blank line after the overall report separates it from the loop

In [ ]:
# Overall summary
overall = PokemonAnalyzer(df)
overall.print_report()
print()

# Per-type summaries
unique_types = sorted(df["primary_type"].unique())
for pokemon_type in unique_types:
    analyzer = PokemonAnalyzer(df, pokemon_type)
    analyzer.print_report()
    print()

---
## Top Pokémon rankings

**Fixes:**
- Comment above the code block, not inline after it
- Blank lines between logical sections

In [ ]:
# Top 5 by base stat total
print("Top 5 Pokemon by Base Stat Total:")
top_total = PokemonAnalyzer(df).top_n("total", 5)
print(top_total.to_string(index=False))

print()

# Top 5 by speed
print("Top 5 Fastest Pokemon:")
top_speed = PokemonAnalyzer(df).top_n("speed", 5)
print(top_speed.to_string(index=False))

---
## Visualization

Create plots to compare stats across Pokémon types.

**Fixes:**
- Multi-line docstring-style block comment replaced with
  section comments (`# ---- Plot 1 ----`)
- Long `plot()` / `hist()` calls wrapped — one kwarg per line
- One statement per line (no semicolons)
- Spaces after every comma

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Plot 1: average base stat total by type ----
ax = axes[0]
type_avg = (
    df.groupby("primary_type")["total"]
    .mean()
    .sort_values(ascending=False)
)
type_avg.plot(
    kind="bar",
    ax=ax,
    color="steelblue",
    edgecolor="black",
    linewidth=0.5,
)
ax.set_title("Avg Base Stat Total by Type")
ax.set_ylabel("Total Stats")
ax.set_xlabel("")
ax.grid(True, alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=45)

# ---- Plot 2: distribution of total stats ----
ax = axes[1]
ax.hist(
    df["total"],
    bins=15,
    color="coral",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.7,
)
mean_total = df["total"].mean()
ax.axvline(
    mean_total,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean={mean_total:.0f}",
)
ax.set_title("Distribution of Base Stat Totals")
ax.set_xlabel("Total Stats")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Radar chart

Compare the stat profiles of the three starter Pokémon
(Bulbasaur, Charmander, Squirtle) using a radar chart.

**Fixes:**
- Multi-line block comment replaced with a proper markdown cell
  (this cell) plus concise inline section comments
- Each list literal on its own lines
- Long `ax.plot()` / `ax.fill()` calls wrapped

In [ ]:
starters = ["bulbasaur", "charmander", "squirtle"]
starter_colors = ["#4CAF50", "#FF5722", "#2196F3"]

# Evenly spaced angles, one per stat
angles = np.linspace(
    0, 2 * np.pi, len(STAT_NAMES), endpoint=False,
).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(
    figsize=(7, 7), subplot_kw={"polar": True},
)

for name, color in zip(starters, starter_colors):
    row = df[df["name"] == name]
    if len(row) == 0:
        continue

    vals = row[STAT_NAMES].values.flatten().tolist()
    vals += vals[:1]  # close the polygon

    ax.plot(
        angles, vals,
        linewidth=2,
        label=name.capitalize(),
        color=color,
    )
    ax.fill(angles, vals, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(STAT_NAMES, fontsize=10)
ax.set_title(
    "Starter Pokemon Stat Comparison",
    y=1.08,
    fontsize=14,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

---
## Save results

Export the processed data to a CSV file.

**Fixes:**
- Long list broken across multiple lines
- One statement per line
- `snake_case` variable name

In [ ]:
export_cols = [
    "name", "id", "primary_type", "total",
    "hp", "attack", "defense",
    "special-attack", "special-defense", "speed",
    "height_m", "weight_kg",
]

df[export_cols].to_csv("pokemon_stats.csv", index=False)

print(f"Saved {len(df)} pokemon to pokemon_stats.csv")
print(f"Columns: {export_cols}")